# Preprocessing

the csv file that should be used in here is found in data\interim\

### meal items 

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import re

In [6]:
import re

# 1. Read the original file
with open('unique_meals.txt', 'r', encoding='utf-8') as f:
    raw_meals = [line.strip() for line in f.readlines() if line.strip()]

# 2. Define the exact splitting logic we used earlier
def extract_single_items(text):
    # Split by +, /, commas, parentheses, or Arabic and/or (و / او / أو)
    parts = re.split(r'[\+\/\(\)\،\,]| او | أو | و ', text)

    # Clean up each part: remove extra punctuation and spaces
    cleaned_parts = [re.sub(r'[^\w\s]', ' ', p).strip() for p in parts]

    # Filter out empty strings or single-letter noise
    return [p for p in cleaned_parts if len(p) > 1]

# 3. Process all meals and collect a flat list of items
all_single_items = []
for meal in raw_meals:
    all_single_items.extend(extract_single_items(meal))

# 4. Remove duplicates by converting to a set, then sort it alphabetically
unique_single_items = sorted(list(set(all_single_items)))

# 5. Save to a new text file
output_filename = 'unique_single_meals.txt'
with open(output_filename, 'w', encoding='utf-8') as f:
    for item in unique_single_items:
        f.write(item + '\n')

# Print a summary of what just happened
print(f"Original composite rows: {len(raw_meals)}")
print(f"Total individual items extracted: {len(all_single_items)}")
print(f"Unique individual items saved: {len(unique_single_items)}")
print(f"Successfully saved to '{output_filename}'!")

Original composite rows: 23360
Total individual items extracted: 53260
Unique individual items saved: 10406
Successfully saved to 'unique_single_meals.txt'!


In [4]:
# 1. Read and clean the meal items
with open('unique_meals.txt', 'r', encoding='utf-8') as f:
    raw_meals = [line.strip() for line in f.readlines() if line.strip()]

def basic_clean(text):
    text = re.sub(r'[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

cleaned_meals = [basic_clean(meal) for meal in raw_meals]

df_meals = pd.DataFrame({
    'original_meal': raw_meals,
    'cleaned_meal': cleaned_meals
})

# 2. Load the GATE Model
# The model supports dimensions [768, 512, 256, 128, 64].
# We set truncate_dim=256 to get high accuracy while keeping the math fast and lightweight.
model_id = 'Omartificial-Intelligence-Space/GATE-AraBert-v1'
print(f"Loading {model_id}...")
model = SentenceTransformer(model_id, truncate_dim=256)

# 3. Generate Embeddings
print(f"Generating Matryoshka embeddings for {len(df_meals)} items...")
embeddings = model.encode(df_meals['cleaned_meal'].tolist(), show_progress_bar=True)

# 4. Cluster the Embeddings
n_clusters = 13
print(f"Clustering into {n_clusters} categories...")
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_meals['cluster_id'] = kmeans.fit_predict(embeddings)

# 5. Review the Results
for cluster_num in range(5):
    print(f"\n--- Cluster {cluster_num} ---")
    sample_items = df_meals[df_meals['cluster_id'] == cluster_num]['original_meal'].head(5).tolist()
    for item in sample_items:
        print(f" - {item}")

# Save the grouped results
df_meals.to_csv('meals_clustered_gate.csv', index=False, encoding='utf-8-sig')
print("\nClustering complete! Saved to 'meals_clustered_gate.csv'")

Loading Omartificial-Intelligence-Space/GATE-AraBert-v1...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating Matryoshka embeddings for 23360 items...


Batches:   0%|          | 0/730 [00:00<?, ?it/s]

Clustering into 13 categories...

--- Cluster 0 ---
 - ياغورت
 - كرواصو
 - ياغورث
 - ياغورت معطر
 - بان شوكو

--- Cluster 1 ---
 - دجاج
 - سمك تونا
 - سمك تونا
 - دجاج
 - طاجين زيتون

--- Cluster 2 ---
 - لحم دجاج
 - لحم طازج
 - لحم طازج
 - لحم أحمر
 - لحم احمر

--- Cluster 3 ---
 - مرق بالخضر
 - مرق جلبانة
 - مرق بالزيتون
 - سلطة كرنب
 - شوربة شعرية

--- Cluster 4 ---
 - عدس
 - شوربة فريك
 - لسان الطير بالمرق
 - سلاطة متنوعة
 - كسكس

Clustering complete! Saved to 'meals_clustered_gate.csv'


In [5]:
# You may need to install plotly if you haven't already:
# !pip install plotly scikit-learn

import plotly.express as px
from sklearn.manifold import TSNE
import pandas as pd

print("Compressing embeddings to 2D using t-SNE... (this may take a minute)")
# t-SNE reduces the 256 dimensions down to 2 dimensions for plotting
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
embeddings_2d = tsne.fit_transform(embeddings)

# Add the X and Y coordinates to our dataframe
df_meals['x'] = embeddings_2d[:, 0]
df_meals['y'] = embeddings_2d[:, 1]

# Convert cluster_id to string so Plotly treats it as a category (distinct colors)
# rather than a continuous color scale
df_meals['cluster_category'] = 'Cluster ' + df_meals['cluster_id'].astype(str)

# Create the interactive scatter plot
fig = px.scatter(
    df_meals,
    x='x',
    y='y',
    color='cluster_category',
    hover_data={'original_meal': True, 'x': False, 'y': False, 'cluster_category': False}, # Clean tooltip
    title="Interactive Semantic Map of Algerian University Meals",
    width=1000,
    height=800
)

# Make the dots slightly transparent so we can see dense areas
fig.update_traces(marker=dict(size=6, opacity=0.7))

# Remove gridlines for a cleaner look
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False, zeroline=False, visible=False),
    yaxis=dict(showgrid=False, zeroline=False, visible=False)
)

# Display the interactive plot in the notebook
fig.show()

Compressing embeddings to 2D using t-SNE... (this may take a minute)
